# NB_lab_test_trends

Builds `adb_caresync_health.gold.lab_test_trends` (one row per test_name per calendar month) per the STTM Silver to Gold v1.0. Driving table: `silver.lab_results` grouped by `test_name`, year, month; LEFT JOIN to `silver.calendar` on the month-start `date_key`. Month-over-month trend via LAG(). Full rebuild.

Notes: `silver.calendar` has no `quarter_name` column, so it is derived as `'Q' || calendar.quarter` (e.g. "Q1"). Exact reload duplicates of `lab_result_id` are removed (latest `insert_timestamp`) before aggregation.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_health.gold;

CREATE OR REPLACE TABLE adb_caresync_health.gold.lab_test_trends AS
WITH lab_dedup AS (
  SELECT *
  FROM adb_caresync_health.silver.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY insert_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    year(test_date)                           AS year,
    month(test_date)                          AS month,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count
  FROM lab_dedup
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(try_divide(abnormal_test_count, total_tests) * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(try_divide(critical_test_count, total_tests) * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                              AS year,
  CAST(w.month AS INT)                             AS month,
  c.month_name,
  concat('Q', c.quarter)                           AS quarter_name,
  CAST(w.total_tests AS INT)                       AS total_tests,
  CAST(w.abnormal_test_count AS INT)               AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)               AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CASE WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
       ELSE CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                              AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN adb_caresync_health.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT);

In [0]:

updated_df = spark.table("adb_caresync_health.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))